# Can a Vision-Language Model Read Your Inspection Images? <sup>🔎 surface defects</sup>

Every vendor pitching AI inspection this year is pitching a vision-language model. The claim is seductive: no training data, no labelling, just point a general-purpose model at your images and ask it questions in English.

This lab tests that claim on **real surface-defect photographs** and gives you the evidence to hold a vendor to. The data is **NEU-CLS** — 1,800 images of hot-rolled steel strip from Northeastern University, six defect classes an inspector would name: *crazing, inclusion, patches, pitted surface, rolled-in scale, scratches*.

We run four experiments, in this order:

1. **Zero-shot CLIP** — ask a vision-language model to name the defect, with no training at all.
2. **A control** — probe the *same frozen features* with a linear classifier. This separates "the model cannot see it" from "the model does not know the word".
3. **A generative VLM writes an inspection report** — on surfaces we know are defective.
4. **The supervised baseline** — a ResNet-18, and a label-scarcity sweep to find where each approach wins.

The conclusion is not "VLMs are useless" and it is not "VLMs solve inspection". It is a specific, defensible engineering recommendation about *how* to use them, and it comes out of numbers you generate in the next fifteen minutes.

> **Runtime.** About 12–18 minutes on a Colab T4. Set *Runtime → Change runtime type → T4 GPU*.
>
> **Licensing note, read before reusing this internally.** NEU-CLS is distributed by its authors for research with a citation request and **no explicit licence grant**. We download it at run time rather than redistributing it. If you intend to build on this inside a commercial setting, get sign-off. Two datasets you might reach for instead — **MVTec AD** and **KolektorSDD** — are CC BY-NC-SA and explicitly forbid commercial use; do not use them for corporate work.

## Setup

In [ ]:
#@title imports
import io, re, urllib.request
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image

np.seterr(all="ignore")
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)
print("device:", device, "| torch", torch.__version__)

In [ ]:
#@title download NEU-CLS
BASE = "https://huggingface.co/datasets/newguyme/neu_cls_caption/resolve/main/data"
root = Path("neu"); root.mkdir(exist_ok=True)
for split in ["train", "test"]:
    dst = root / f"{split}.parquet"
    if not dst.exists():
        urllib.request.urlretrieve(f"{BASE}/{split}-00000-of-00001.parquet", dst)

train_df = pd.read_parquet(root / "train.parquet")
test_df  = pd.read_parquet(root / "test.parquet")
CLASSES  = sorted(train_df.label_str.unique())
print(f"train {len(train_df)} | test {len(test_df)} | {len(CLASSES)} classes")
print(train_df.groupby(["label", "label_str"]).size().rename("n").to_frame().T.to_string())

def to_images(df):
    return [Image.open(io.BytesIO(r["bytes"])).convert("RGB") for r in df["image"]]

train_imgs, test_imgs = to_images(train_df), to_images(test_df)
y_train, y_test = train_df.label.values, test_df.label.values
print(f"image size {train_imgs[0].size} | chance accuracy {100/len(CLASSES):.1f}%")

In [ ]:
#@title what the inspector sees
fig, axes = plt.subplots(len(CLASSES), 6, figsize=(11, 11))
for r, c in enumerate(CLASSES):
    idx = np.where(y_train == CLASSES.index(c))[0][:6]
    for a, i in zip(axes[r], idx):
        a.imshow(train_imgs[i], cmap="gray"); a.axis("off")
    axes[r, 0].set_ylabel(c, rotation=0, ha="right", va="center", fontsize=9)
    axes[r, 0].axis("on"); axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
fig.suptitle("NEU-CLS — six defect classes of hot-rolled steel strip", y=0.995)
plt.tight_layout(); plt.show()

Look at these the way a mill inspector would. *Crazing* is a fine crack network. *Rolled-in scale* is oxide pressed into the surface during rolling. *Pitted surface* is dense corrosion pitting. *Inclusion* is foreign material rolled into the strip.

These are not obscure words in a steelworks. They are entirely absent from the internet captions a general vision-language model was trained on. Hold that thought.

## Experiment 1 — zero-shot CLIP

CLIP was trained to match images to captions across 400M web image–text pairs. To classify without any training you write one caption per class and pick whichever the image matches best. That is the whole vendor pitch, and it genuinely works well on everyday categories.

We try four prompt styles, from bare class names up to a domain-specific template, because "you just need better prompts" is the first thing you will be told when it underperforms.

In [ ]:
#@title CLIP zero-shot, four prompt styles
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
pretty = lambda c: c.replace("_", " ").replace("-", " ")

# CLIPModel.get_image_features / get_text_features return different things on
# different transformers releases: a projected tensor on some, a raw
# BaseModelOutputWithPooling on others - and in the latter case pooler_output is
# ALREADY projected, so projecting it again raises a 768x512 matmul error.
# Shape cannot disambiguate either: for ViT-B/32 the text hidden size and
# projection_dim are both 512. So we never call those methods. We run the towers
# and their projections explicitly, which is unambiguous on every release:
#   vision_model.pooler_output -> 768  --visual_projection-->  512
#   text_model.pooler_output   -> 512  --text_projection  -->  512
# Verified bit-identical to get_*_features where those work (max abs diff 0.0).

@torch.no_grad()
def image_features(images, bs=64):
    out = []
    for i in range(0, len(images), bs):
        px = proc(images=images[i:i+bs], return_tensors="pt").to(device)
        pooled = clip.vision_model(pixel_values=px["pixel_values"]).pooler_output
        f = clip.visual_projection(pooled)
        out.append((f / f.norm(dim=-1, keepdim=True)).cpu())
    return torch.cat(out)

@torch.no_grad()
def text_features(prompts):
    tk = proc(text=prompts, return_tensors="pt", padding=True).to(device)
    pooled = clip.text_model(input_ids=tk["input_ids"],
                             attention_mask=tk.get("attention_mask")).pooler_output
    f = clip.text_projection(pooled)
    return (f / f.norm(dim=-1, keepdim=True)).cpu()

# fail loudly here rather than silently producing garbage similarities later
_probe   = image_features(test_imgs[:2])
_probe_t = text_features(["a scratch", "a clean surface"])
D = clip.config.projection_dim
assert _probe.shape[1]   == D, f"image embedding is {_probe.shape[1]}-d, expected {D}"
assert _probe_t.shape[1] == D, f"text embedding is {_probe_t.shape[1]}-d, expected {D}"
print(f"CLIP embeddings OK: image {tuple(_probe.shape)}, text {tuple(_probe_t.shape)}, projection_dim={D}")

E_test  = image_features(test_imgs)
E_train = image_features(train_imgs)

PROMPTS = {
    "bare class name":       [pretty(c) for c in CLASSES],
    "a photo of {c}":        [f"a photo of {pretty(c)}" for c in CLASSES],
    "metal defect template": [f"a photo of a metal surface with {pretty(c)} defect" for c in CLASSES],
    "steel inspection":      [f"a close-up photograph of hot-rolled steel showing {pretty(c)}" for c in CLASSES],
}

rows, best = {}, None
for name, ps in PROMPTS.items():
    pred = (E_test @ text_features(ps).T).argmax(1).numpy()
    acc = 100 * (pred == y_test).mean()
    rows[name] = acc
    if best is None or acc > best[0]: best = (acc, name, pred)
print(pd.Series(rows, name="zero-shot accuracy %").round(1).to_string())
print(f"\nchance = {100/len(CLASSES):.1f}%   best = {best[1]} at {best[0]:.1f}%")
zs_acc, zs_pred = best[0], best[2]

In [ ]:
#@title where does it actually put the images?
print("per-class recall, best prompt:\n")
for i, c in enumerate(CLASSES):
    m = y_test == i
    print(f"  {c:18s} recall {100*(zs_pred[m]==i).mean():5.1f}%   mostly predicted: {CLASSES[np.bincount(zs_pred[m], minlength=len(CLASSES)).argmax()]}")
print(f"\nprediction histogram over all {len(y_test)} test images:")
for c, n in Counter(CLASSES[p] for p in zs_pred).most_common():
    print(f"  {c:18s} {n:4d}  ({100*n/len(y_test):4.1f}%)")

cm = np.zeros((len(CLASSES), len(CLASSES)), int)
for t, p in zip(y_test, zs_pred): cm[t, p] += 1
fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=90, fontsize=8)
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES, fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"CLIP zero-shot ({zs_acc:.1f}%)")
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        if cm[i,j]: ax.text(j, i, cm[i,j], ha="center", va="center", fontsize=7,
                            color="white" if cm[i,j] > cm.max()/2 else "black")
fig.colorbar(im); plt.tight_layout(); plt.show()

Barely above chance, and prompt engineering buys only a few points. Worse than the headline number is the *shape* of the failure: the predictions pile into one or two classes and several defect types get **zero** recall. The model is not making subtle mistakes — it has no usable notion of these categories and is effectively guessing a favourite.

Before concluding anything, we have to rule out the boring explanation.

## Experiment 2 — the control that matters

There are two very different reasons zero-shot could fail:

- **(a)** the vision encoder cannot represent these textures at all, or
- **(b)** the encoder sees them fine, but the *text side* has never learned what "rolled-in scale" means.

These have opposite consequences. If (a), a VLM is the wrong tool for inspection. If (b), the features are valuable and only the language interface is broken.

The test is simple: freeze the encoder, throw the text tower away, and fit a plain logistic regression on the image features. If that works, the information was there all along.

In [ ]:
#@title linear probe on the same frozen CLIP features
from sklearn.linear_model import LogisticRegression

probe = LogisticRegression(max_iter=3000, C=10).fit(E_train.numpy(), y_train)
probe_acc = 100 * probe.score(E_test.numpy(), y_test)
print(f"CLIP zero-shot (text interface) : {zs_acc:5.1f}%")
print(f"CLIP linear probe (same features): {probe_acc:5.1f}%")
print(f"chance                           : {100/len(CLASSES):5.1f}%")
print(f"\n-> the frozen encoder carries {probe_acc:.0f}% worth of information about these defects.")
print("   The eyes work. The vocabulary is what failed.")

That is the finding to take into a vendor meeting. The visual features are strong; the zero-shot text interface is what collapses. "Prompt it better" cannot fix a word the text encoder never learned — but you can bypass the text encoder entirely.

## Experiment 3 — let a generative VLM write the report

CLIP only scores captions. A *generative* VLM writes prose, which is what an inspection assistant would actually be asked to do. So we ask a small instruction-tuned VLM two things about images we **know** contain defects:

1. a closed question — "does this surface show any defect?"
2. an open request — "describe the condition of this metal surface for an inspection report."

Every image in the main test is defective, so there is no correct answer that says otherwise. We also ask the same question about **controls** whose defect texture has been destroyed — flat grey fields and heavily blurred images — because a denial rate measured only on positives cannot tell a model that misses defects apart from one that always answers the same thing.

One detail that matters for honesty: deciding *automatically* whether a reply denies the defect is itself error-prone. A naive keyword rule flags "the surface is **not** uniformly smooth" as a denial when it is the opposite. We therefore count a reply as a denial only when it asserts a clean surface **and** never names any damage.

In [ ]:
#@title a small VLM answers a yes/no defect question
from transformers import AutoProcessor
try:                                              # newer releases prefer this name
    from transformers import AutoModelForImageTextToText as VLMClass
except ImportError:                               # older releases only have the old one
    from transformers import AutoModelForVision2Seq as VLMClass

VLM_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
vproc = AutoProcessor.from_pretrained(VLM_ID)
vlm = VLMClass.from_pretrained(
    VLM_ID, dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()

@torch.no_grad()
def ask(img, prompt, max_new_tokens=70):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    tpl  = vproc.apply_chat_template(msgs, add_generation_prompt=True)
    inp  = vproc(text=tpl, images=[img], return_tensors="pt").to(device)
    out  = vlm.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
    return vproc.batch_decode(out, skip_special_tokens=True)[0].split("Assistant:")[-1].strip()

# Detecting "the model denied the defect" needs care: a naive /\bnot\b|smooth/ fires on
# defect-AFFIRMING prose such as "the surface is not uniformly smooth". We therefore require
# an explicit denial phrase AND the absence of any damage word.
DENIAL = re.compile(r"^\s*no\b|\bno (?:visible|significant|obvious|apparent|sign|signs|defect|damage|corrosion|crack)"
                    r"|free (?:of|from)|without any|appears? (?:clean|smooth|uniform|undamaged)"
                    r"|in good condition|well[- ]maintained|no discernible", re.I)
DAMAGE = re.compile(r"rust|corros|crack|craz|pit(?:ted|ting)|scratch|dent|defect|damage|flaw|scale|"
                    r"inclusion|patch|blemish|wear|weathered|rough|uneven|irregular", re.I)

# Damage words also appear INSIDE negations ("no visible signs of rust"), so we strip
# negated clauses before asking whether the model actually named any damage.
NEG_SPAN = re.compile(r"(?:\bno\b|\bnot\b|\bnor\b|free (?:of|from)|without)[^.;]*", re.I)

def denies_defect(text):
    """True only if the model asserts a clean surface and never names damage outside a negation."""
    return bool(DENIAL.search(text)) and not DAMAGE.search(NEG_SPAN.sub(" ", text))

# the detector is itself a model of language, so we test it before trusting it
_CASES = [
    ("The surface is in good condition, with no visible signs of rust or damage.", True),
    ("The surface appears corroded, with visible signs of rust. It is not uniformly coated.", False),
    ("No.", True), ("Yes.", False),
    ("Yes, the surface is rough, not smooth.", False),
]
assert all(denies_defect(t) == e for t, e in _CASES), "denial detector failed its own test cases"
print("denial detector: passed", len(_CASES), "test cases")

QUESTION = "Does this metal surface show any defect? Answer only yes or no."
rs = np.random.RandomState(0)

# POSITIVES: 30 images that every one of them contains a defect
answers = []
for c in range(len(CLASSES)):
    for i in rs.choice(np.where(y_test == c)[0], 5, replace=False):
        answers.append((CLASSES[c], ask(test_imgs[i], QUESTION, 8)))

# CONTROLS: images with the defect texture destroyed. Without these we could not tell
# "the model misses defects" apart from "the model answers the same thing regardless".
from PIL import ImageFilter
controls  = [Image.new("RGB", (200, 200), v) for v in (100, 128, 160)]
controls += [test_imgs[i].filter(ImageFilter.GaussianBlur(12))
             for i in rs.choice(len(test_imgs), 7, replace=False)]
ctrl_answers = [ask(im, QUESTION, 8) for im in controls]

den_pos  = sum(1 for _, a in answers      if denies_defect(a))
den_ctrl = sum(1 for a    in ctrl_answers if denies_defect(a))
print(f"defective images     : {len(answers):3d}   denied a defect: {den_pos:3d}  ->  {100*den_pos/len(answers):.0f}%  (these are false negatives)")
print(f"flat/blurred controls: {len(ctrl_answers):3d}   denied a defect: {den_ctrl:3d}  ->  {100*den_ctrl/len(ctrl_answers):.0f}%  (denial here is CORRECT)")
print("\nanswer distribution on defective images:")
for a, k in Counter(a.lower()[:20] for _, a in answers).most_common(6): print(f"  {k:3d}x  {a!r}")
print("answer distribution on controls:")
for a, k in Counter(a.lower()[:20] for a in ctrl_answers).most_common(6): print(f"  {k:3d}x  {a!r}")
gap = 100*den_ctrl/len(ctrl_answers) - 100*den_pos/len(answers)
print(f"\ndenial-rate gap (control - defective): {gap:+.0f} points")
print("  >0 means the answer does depend on the image; ~0 means it does not depend on the image at all.")

In [ ]:
#@title now ask it to write an inspection report
for c in [CLASSES.index(x) for x in ["crazing", "pitted_surface", "rolled-in_scale"] if x in CLASSES]:
    i = int(rs.choice(np.where(y_test == c)[0], 1)[0])
    report = ask(test_imgs[i], "Describe the condition of this metal surface for an inspection report.")
    flag = "   <-- FALSE NEGATIVE" if denies_defect(report) else "   (names the damage)"
    fig, ax = plt.subplots(figsize=(2.2, 2.2)); ax.imshow(test_imgs[i], cmap="gray"); ax.axis("off")
    ax.set_title(f"true class: {CLASSES[c]}", fontsize=9); plt.show()
    print(f"VLM:{flag}\n  {report}\n")

This is the single most useful thing in the session for an integrity engineer — but only because of the control row.

**Read the two rates together, never the first one alone.** Had we asked only about defective images, a model that answers "no" to *every* grey texture would post a spectacular false-negative rate and we would have concluded it "errs toward declaring surfaces sound". The real finding in that case would be worse and quite different: that the answer carries no information about the image at all. The controls — flat grey fields and images blurred until the defect texture is gone — are what separate those two stories. Denying a defect on those is the *correct* answer, so the gap between the two denial rates is the only evidence that the model is looking at anything.

This is the same discipline as Experiment 2. A number without its control is a number you cannot interpret, and "we only tested it on positives" is the most common way an impressive-looking inspection benchmark turns out to mean nothing.

Whatever the rates, the free-text cell below is the part to show a manager. The model does not say "I am unsure" or "this texture is unfamiliar" — it produces **fluent, confident, professional-sounding inspection prose**. When that prose is wrong in the direction of declaring a defective surface sound, it is the worst available failure: a hedge you can catch in review, a confident false negative you cannot.

Note the honest caveat on model size: this is a 500M-parameter model, chosen so the lab runs on a free T4. A larger VLM writes better prose. But prose quality is not the failure — the *vocabulary gap* measured in Experiments 1 and 2 is, and that does not close by scaling into a domain the training captions never covered. Re-run with a bigger model and check; that is the exercise.

## Experiment 4 — the baseline, and when each approach wins

A supervised ResNet-18 on 1,440 labelled images is the incumbent. It is unglamorous and it took ten minutes to set up. Then the question that actually decides your project plan: **how many labels do you need before supervised training beats using the frozen VLM features?**

Every ResNet run gets the **same 300 optimiser steps** regardless of label count. Equalising epochs instead would confound the comparison — 60 labels would get 22× fewer gradient updates than 1,440, and you would be measuring the training budget rather than the labels. Each point is the **mean of 3 seeds** with error bars; at 60 labels a single seed swings by tens of points, and a curve drawn through single-seed numbers can easily be non-monotonic for no real reason.

In [ ]:
#@title ResNet-18 fine-tune vs CLIP probe, as labels get scarce
import torchvision
import torchvision.transforms as T

tf = T.Compose([T.Resize((224, 224)), T.ToTensor(),
                T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
Xtr_i = torch.stack([tf(im) for im in train_imgs]); ytr_i = torch.tensor(y_train)
Xte_i = torch.stack([tf(im) for im in test_imgs]);  yte_i = torch.tensor(y_test)

TRAIN_STEPS = 300      # fixed optimiser steps, so the sweep varies LABELS and nothing else

def resnet_accuracy(idx, steps=TRAIN_STEPS, seed=0):
    torch.manual_seed(seed)
    m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(512, len(CLASSES)); m = m.to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=1e-4)
    dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xtr_i[idx], ytr_i[idx]),
                                     batch_size=32, shuffle=True)
    m.train(); done = 0
    while done < steps:                       # equal gradient budget at every label count
        for xb, yb in dl:
            loss = nn.functional.cross_entropy(m(xb.to(device)), yb.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
            done += 1
            if done >= steps: break
    m.eval(); preds = []
    with torch.no_grad():
        for i in range(0, len(Xte_i), 64):
            preds.append(m(Xte_i[i:i+64].to(device)).argmax(1).cpu())
    return 100 * (torch.cat(preds) == yte_i).float().mean().item()

SEEDS = 3        # a single seed on 60 labels is far too noisy to draw a curve through
rs2 = np.random.RandomState(0)
rows = []
for per_class in [10, 30, 240]:
    ps, rsn = [], []
    for s_ in range(SEEDS):
        rr = np.random.RandomState(100 + s_)
        idx = np.concatenate([rr.choice(np.where(y_train == c)[0], per_class, replace=False) for c in range(len(CLASSES))])
        ps.append(100 * LogisticRegression(max_iter=3000, C=10).fit(E_train.numpy()[idx], y_train[idx]).score(E_test.numpy(), y_test))
        rsn.append(resnet_accuracy(idx, seed=s_))
    rows.append({"labels": per_class*len(CLASSES),
                 "CLIP linear probe %": np.mean(ps), "probe sd": np.std(ps),
                 "ResNet-18 fine-tune %": np.mean(rsn), "resnet sd": np.std(rsn),
                 "CLIP zero-shot %": zs_acc})
res = pd.DataFrame(rows).set_index("labels")
print(res.round(1).to_string())

plt.figure(figsize=(6.5, 3.4))
plt.errorbar(res.index, res["CLIP linear probe %"], yerr=res["probe sd"], fmt="o-", capsize=3, label="CLIP linear probe")
plt.errorbar(res.index, res["ResNet-18 fine-tune %"], yerr=res["resnet sd"], fmt="s-", capsize=3, label="ResNet-18 fine-tune")
plt.axhline(zs_acc, color="k", ls="--", lw=1, label=f"CLIP zero-shot ({zs_acc:.0f}%)")
plt.axhline(100/len(CLASSES), color="grey", ls=":", lw=1, label="chance")
plt.xscale("log"); plt.xlabel("labelled training images"); plt.ylabel("test accuracy %")
plt.title(f"what your labelling budget buys (mean of {SEEDS} seeds)"); plt.grid(alpha=0.3); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Where this leaves you

**The recommendation, in one line: use a vision-language model as a frozen feature extractor, never as an oracle.**

The evidence you just generated, in the order it matters:

1. **Zero-shot classification of named defects is close to chance**, and prompt engineering does not rescue it. The predictions collapse onto a favourite class and several defect types score zero recall.
2. **The same frozen encoder supports a strong linear probe.** So the visual information is present — what fails is the text interface, which never learned mill vocabulary. This is the distinction to insist on when a vendor shows you a zero-shot demo on cats and bottles.
3. **A generative VLM writes confident, fluent, wrong inspection reports.** Compare its denial rate on defective images against the flat/blurred controls before claiming a *direction* of failure: if the two rates are close, the answer is not image-dependent at all, which is the more damning result. Either way the prose is authoritative and unhedged, which is what makes it dangerous in a report.
4. **With labels, supervised training wins outright.** With very few labels, the frozen probe is competitive or better — which tells you where to spend a labelling budget rather than whether to have one.

**What this does not show.** We tested one CLIP checkpoint and one small generative VLM on one dataset of one material. A larger VLM, or one pretrained on industrial imagery, would score better — the lab is a method for interrogating a claim, not a universal verdict. The honest generalisation is narrower and more useful: *a model's zero-shot vocabulary is bounded by its training captions, and your plant's vocabulary is probably not in there.*

**Worth trying next.** Swap in a larger VLM and see how much of the gap closes. Fine-tune CLIP's text tower on 200 captions written by an inspector and watch zero-shot jump — that is the cheapest real fix. Or take the linear probe and check its confusion against the classes an inspector says are genuinely hard to tell apart; if the model confuses the same pairs a human does, that is a good sign rather than a bad one.

**References.** Radford et al., *Learning Transferable Visual Models From Natural Language Supervision* (CLIP, 2021), [arXiv:2103.00020](https://arxiv.org/abs/2103.00020) · Song & Yan, *A noise robust method based on completed local binary patterns for hot-rolled steel strip surface defects* (2013) — the NEU-CLS source · Marafioti et al., *SmolVLM* (2025), [arXiv:2504.05299](https://arxiv.org/abs/2504.05299)